# 2.2 — Bond risk, curve buckets and spread measures

This lab starts from the foundations bond and market introduced before lesson 2.2. It builds each risk calculation locally and contains no later-stage holdings.


## Analyst program: qualified rate and spread risk

Parallel curve DV01 is checked by bumping continuously compounded curve zeros. Yield duration and convexity are separate quantities because they use a single yield convention. The second-order curve estimate below is measured from repricing and compared with the full 25 bp shock.

In [ ]:
from datetime import date
import copy, json, math
from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.valuations.instruments import price_instrument
from _shared.analyst_book import build_market, instruments
AS_OF = date(2025, 1, 15)

market = build_market('foundations')
curve = market.get_discount('USD-OIS')
analysis_bond = instruments('foundations')['USD-CORP']
# ASW constructs a new settlement-to-maturity leg, which has a short front stub.
analysis_bond['instrument']['spec']['cashflow_spec']['fixed']['stub'] = 'short_front'
bond = json.dumps(analysis_bond)
result = price_instrument(bond, market, AS_OF, metrics=['dv01', 'bucketed_dv01', 'duration_mod', 'convexity', 'cs01', 'z_spread', 'i_spread', 'asw_par'])
values = {}
for bp in (-1.0, 0.0, 1.0, 25.0):
    bumped = MarketContext.from_json(market.to_json())
    bumped.insert(DiscountCurve('USD-OIS', AS_OF,
        [(t, df * math.exp(-bp * 1e-4 * t)) for t, df in zip(curve.knots, curve.dfs)], interp='log_linear'))
    values[bp] = price_instrument(bond, bumped, AS_OF).price
dv01 = (values[1.0] - values[-1.0]) / 2  # native signed change for +1 bp
assert abs(dv01 - result.get_metric('dv01')) < 0.01
actual = values[25.0] - values[0.0]
linear = 25 * dv01
quadratic = linear + 0.5 * 25 ** 2 * (values[1.0] + values[-1.0] - 2 * values[0.0])
assert abs(actual - quadratic) < abs(actual - linear)
assert result.get_metric('cs01::USD-CORP') is not None
assert any(key.startswith('bucketed_dv01::USD-OIS::') for key in result.metric_keys())
print({'actual_pnl': actual, 'linear': linear, 'quadratic': quadratic, 'yield_convexity': result.get_metric('convexity')})

## Yield duration and scaled convexity

In [ ]:
from datetime import date
import json
from finstack_quant.valuations.instruments import price_instrument
from _shared.analyst_book import build_market, instruments
AS_OF = date(2025, 1, 15)
contract=instruments('foundations')['USD-CORP'];market=build_market('foundations')
risk=price_instrument(json.dumps(contract),market,AS_OF,metrics=['ytm','duration_mac','duration_mod','convexity'])
y=risk.get_metric('ytm');dmac=risk.get_metric('duration_mac');dmod=risk.get_metric('duration_mod')
assert abs(dmac-dmod*(1+y/2))<1e-8
print({'Macaulay_years':dmac,'modified_years':dmod,'native_convexity':risk.get_metric('convexity'),'decimal_yield_convexity':100*risk.get_metric('convexity')})


## Benchmark and floating-rate spread measures

In [ ]:
from datetime import date, timedelta
import copy,json
from finstack_quant.valuations.instruments import price_instrument
from finstack_quant.core.market_data import ForwardCurve, ScalarTimeSeries
from _shared.analyst_book import build_market, instruments
from _shared.instrument_fixtures import floating_bond,instrument_envelope
AS_OF = date(2025, 1, 15)
market=build_market('foundations')
# This flat term-SOFR input is local to the FRN comparison; the shared rates book begins in 2.3.
market.insert(ForwardCurve.flat('USD-SOFR-3M', 0.25, AS_OF, 0.05))
first_fixing = date(2024, 1, 1)
term_sofr_fixings = [(first_fixing + timedelta(days=offset), 0.05) for offset in range((AS_OF - first_fixing).days + 1)]
market.insert_series(ScalarTimeSeries('FIXING:USD-SOFR-3M', term_sofr_fixings))
corporate=instruments('foundations')['USD-CORP']
corporate['instrument']['spec']['cashflow_spec']['fixed']['stub']='short_front'
corporate['instrument']['spec']['instrument_pricing_overrides']={'market_quotes':{'quoted_clean_price':99.0}}
r=price_instrument(json.dumps(corporate),market,AS_OF,metrics=['ytm','i_spread','z_spread','asw_par'])
# A same-currency synthetic government yield is an explicit benchmark, not a native g_spread metric.
government_yield=0.04
g_spread_bp=(r.get_metric('ytm')-government_yield)*1e4
_,frn=floating_bond(0)
frn_result=price_instrument(json.dumps(instrument_envelope(frn)),market,AS_OF,metrics=['discount_margin'])
assert all(r.get_metric(k) is not None for k in ['i_spread','z_spread','asw_par'])
print({'government_yield_decimal':government_yield,'G_spread_bp':g_spread_bp,'I_spread_bp':1e4*r.get_metric('i_spread'),'Z_spread_bp':1e4*r.get_metric('z_spread'),'ASW_par_bp':1e4*r.get_metric('asw_par'),'FRN_discount_margin_decimal':frn_result.get_metric('discount_margin')})


## Seven-year bond bucket support

In [ ]:
from datetime import date
import json,math
from finstack_quant.core.market_data import DiscountCurve
from finstack_quant.valuations.instruments import price_instrument
from _shared.analyst_book import build_market, instruments
AS_OF = date(2025, 1, 15)
market=build_market('foundations').insert(DiscountCurve('USD-OIS',AS_OF,[(0,1)]+[(t,math.exp(-.04*t)) for t in (2,5,10,30)],interp='log_linear'))
contract=instruments('foundations')['USD-CORP'];contract['instrument']['spec']['maturity']='2032-01-15'
r=price_instrument(json.dumps(contract),market,AS_OF,metrics=['dv01','bucketed_dv01'])
buckets={key:r.get_metric(key) for key in r.metric_keys() if key.startswith('bucketed_dv01::USD-OIS::')}
assert len([v for v in buckets.values() if abs(v)>0.01])>1
assert r.get_metric('dv01')<0 and sum(buckets.values())<0
print({'curve_pillars_years':[2,5,10,30],'parallel_DV01_USD_per_bp':r.get_metric('dv01'),'native_key_rate_buckets':buckets})


## Bond z-spread CS01 sign and key

In [ ]:
from datetime import date
import copy,json
from finstack_quant.valuations.instruments import price_instrument
from _shared.analyst_book import build_market, instruments
AS_OF = date(2025, 1, 15)
market=build_market('foundations');contract=instruments('foundations')['USD-CORP']
r=price_instrument(json.dumps(contract),market,AS_OF,metrics=['cs01','z_spread'])
z=r.get_metric('z_spread')
values={}
for bp in (-1,0,1):
 bumped=copy.deepcopy(contract)
 bumped['instrument']['spec']['instrument_pricing_overrides']={'market_quotes':{'quoted_z_spread':z+bp*1e-4}}
 values[bp]=price_instrument(json.dumps(bumped),market,AS_OF,metrics=['dirty_price']).get_metric('dirty_price')
native=r.get_metric('cs01::USD-CORP');forward=values[1]-values[0];central=(values[1]-values[-1])/2
assert native<0 and abs(native-forward)<.01
print({'bond_zspread_CS01_USD_per_bp':native,'native_forward_difference':forward,'symmetric_difference':central,'convexity_difference':forward-central})


## Bullet versus amortizing duration

In [ ]:
import json
from datetime import date
from decimal import Decimal
from finstack_quant.cashflows.builder import CashFlowSchedule, FixedCouponSpec, ScheduleParams, AmortizationSpec
from finstack_quant.core.money import Money
from finstack_quant.valuations.instruments import bond_from_cashflows_json, price_instrument
from _shared.analyst_book import build_market, instruments
AS_OF = date(2025, 1, 15)

market = build_market('foundations')
schedule = (CashFlowSchedule.builder().principal(Money(2_000_000.0, 'USD'), date(2024, 1, 15), date(2030, 1, 15))
    .amortization(AmortizationSpec.linear_to(Money(0.0, 'USD')))
    .fixed_cf(FixedCouponSpec(Decimal('0.055'), ScheduleParams.semiannual_30360()))
    .build())
amortizer = bond_from_cashflows_json('LESSON-AMORTIZER', schedule.to_json(), 'USD-OIS')
amortized = price_instrument(amortizer, market, AS_OF, metrics=['duration_mod', 'ytm'])
bullet = price_instrument(json.dumps(instruments('foundations')['USD-CORP']), market, AS_OF, metrics=['duration_mod'])
assert amortized.price > 0
assert amortized.get_metric('duration_mod') < bullet.get_metric('duration_mod')
print({'amortizer_duration': amortized.get_metric('duration_mod'), 'bullet_duration': bullet.get_metric('duration_mod')})
